<a href="https://colab.research.google.com/github/Mahammad-Haroon/Data-Analysis-Projects/blob/main/Outlier_Detection_Revenue_Audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Detecting Outliers in Revenue Transactions

Run Dataset Generation Code

In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

# Retail Transactions
retail_amt = np.random.normal(loc=1750, scale=500, size=100)
retail_amt = np.clip(retail_amt, 500, 3000)

# Wholesale Transactions
wholesale_amt = np.random.uniform(40000, 80000, size=8)

# Extreme Outliers (Errors)
error_amt = [350000, 420000]

all_amounts = np.concatenate([retail_amt, wholesale_amt, error_amt])
np.random.shuffle(all_amounts)

df = pd.DataFrame({
    'Transaction_ID': [f'TXN_{i+1000}' for i in range(len(all_amounts))],
    'Customer_ID': [f'CUST_{np.random.randint(1, 50)}' for _ in range(len(all_amounts))],
    'Transaction_Amount': all_amounts.round(2),
    'Customer_Type': ['Retail' if x < 10000 else 'Wholesale' for x in all_amounts],
    'Payment_Method': np.random.choice(['Credit Card', 'UPI', 'Net Banking'], size=len(all_amounts)),
    'Order_Date': pd.date_range(start='2023-10-01', periods=len(all_amounts), freq='D')
})

df.head()

,Transaction_ID,Customer_ID,Transaction_Amount,Customer_Type,Payment_Method,Order_Date
0,TXN_1000,CUST_21,1221.14,Retail,Net Banking,2023-10-01
1,TXN_1001,CUST_48,2021.28,Retail,UPI,2023-10-02
2,TXN_1002,CUST_20,1010.74,Retail,Net Banking,2023-10-03
3,TXN_1003,CUST_8,1870.98,Retail,Credit Card,2023-10-04
4,TXN_1004,CUST_7,2482.82,Retail,Credit Card,2023-10-05


Z-Score Outlier Detection

In [2]:
from scipy.stats import zscore

df["Z_Score"] = zscore(df["Transaction_Amount"])

z_outliers = df[np.abs(df["Z_Score"]) > 3]

z_count = len(z_outliers)
z_max = z_outliers["Transaction_Amount"].max()
z_wholesale_flag = (z_outliers["Customer_Type"] == "Wholesale").any()

print(z_count)
print(z_max)
print(z_wholesale_flag)

2
420000.0
True


IQR Outlier Detection

In [3]:
Q1 = df["Transaction_Amount"].quantile(0.25)
Q3 = df["Transaction_Amount"].quantile(0.75)
IQR = Q3 - Q1

lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

iqr_outliers = df[(df["Transaction_Amount"] < lower_fence) | (df["Transaction_Amount"] > upper_fence)]

iqr_count = len(iqr_outliers)
iqr_max = iqr_outliers["Transaction_Amount"].max()
iqr_wholesale_flag = (iqr_outliers["Customer_Type"] == "Wholesale").any()

print(iqr_count)
print(iqr_max)
print(iqr_wholesale_flag)

10
420000.0
True


Generate Comparison Table

In [4]:
comparison_table = pd.DataFrame({
    "Method": ["Z-Score", "IQR"],
    "No_of_Outliers": [z_count, iqr_count],
    "Max_Outlier_Amount": [z_max, iqr_max],
    "Wholesale_Flagged_(True/False)": [z_wholesale_flag, iqr_wholesale_flag]
})

comparison_table

,Method,No_of_Outliers,Max_Outlier_Amount,Wholesale_Flagged_(True/False)
0,Z-Score,2,420000.0,True
1,IQR,10,420000.0,True


Apply Data Cleaning Policy

In [5]:
clean_df = df.copy()

clean_df = clean_df[clean_df["Transaction_Amount"] <= 250000]

clean_df.loc[
    (clean_df["Customer_Type"] == "Wholesale") &
    (clean_df["Transaction_Amount"] > 80000),
    "Transaction_Amount"
] = 80000

Add IQR Outlier Column

In [6]:
clean_df["Is_Outlier_IQR"] = (
    (clean_df["Transaction_Amount"] < lower_fence) |
    (clean_df["Transaction_Amount"] > upper_fence)
)

Create Before vs After Table

In [7]:
summary_table = pd.DataFrame({
    "Metric": ["Mean", "Max"],
    "Before_Cleaning": [
        df["Transaction_Amount"].mean(),
        df["Transaction_Amount"].max()
    ],
    "After_Cleaning": [
        clean_df["Transaction_Amount"].mean(),
        clean_df["Transaction_Amount"].max()
    ]
})

summary_table

,Metric,Before_Cleaning,After_Cleaning
0,Mean,12756.948909,5863.559074
1,Max,420000.000000,77716.390000
